# TTM × SIX — G1 Colab môi trường v2: preflight rồi smoke kỹ thuật

Sau STOP đúng chuẩn ở phiên 2026-11-09, hợp đồng môi trường được sửa bằng dữ liệu đọc thực tế (Python 3.13.16, torch 2.14.1+cu130, transformers 5.22.1, huggingface-hub 1.33.0, tokenizers 0.23.3, safetensors 0.8.0, Tesla T4). Tất cả mã nguồn/nguồn Qwen được ghim bằng SHA.

**Ô 1**: lấy script ~7 KB, kiểm Git blob, biên dịch, tiền kiểm môi trường. **Không tải mô hình.** Nếu STOP, không chạy ô 2.

**Ô 2**: chỉ khi ô 1 PASS, chủ sở hữu chủ động chạy ô này để tải ~1,5 GB trọng số Qwen và thực hiện đúng một lượt kiểm thử tensor tổng hợp. Marker một lần ngăn chạy lặp lại trong cùng phiên. Không huấn luyện, không dùng corpus hoặc TEST, không ghi checkpoint hoặc Google Drive. Giữ receipt khi PASS/FAIL. Không dùng notebook này để tự cấp science lock.


In [ ]:
# G1 v2 — Verify script SHA, compile, preflight. No Qwen weight download.
import ast, hashlib, json, pathlib, py_compile, subprocess, sys, urllib.request
P_COMMIT="66cbc40ee65072c375bccdc3eebfceb76236174e"
P_BLOB="c98811e167696f86e94e3da9d01f3b9f020c3cc6"
P_SOURCE="scripts/research/six_ttm/g1_qwen06_zero_science_smoke_colab_env_20261109_v2.py"
P_ROOT=pathlib.Path("/content/ttm_six_g1")
P_URL=f"https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/{P_COMMIT}/{P_SOURCE}"
if not pathlib.Path("/content").is_dir():
    raise RuntimeError("STOP_NOT_COLAB")
source=urllib.request.urlopen(P_URL,timeout=45).read()
git_blob=hashlib.sha1(b"blob "+str(len(source)).encode()+b"\0"+source).hexdigest()
print("G1_V2_SCRIPT_BLOB="+git_blob)
if git_blob!=P_BLOB:
    raise RuntimeError("STOP_SCRIPT_PIN_MISMATCH")
ast.parse(source.decode("utf-8"),filename=P_SOURCE)
P_ROOT.mkdir(parents=True,exist_ok=True)
script=P_ROOT/"g1_qwen06_zero_science_smoke_colab_env_20261109_v2.py"
script.write_bytes(source)
py_compile.compile(str(script),doraise=True)
print("G1_V2_STATIC_COMPILE=PASS")
preflight=subprocess.run([sys.executable,str(script),"--phase","preflight"],check=False)
print("G1_V2_PREFLIGHT_EXIT="+str(preflight.returncode))
receipt_path=P_ROOT/"G1_PREFLIGHT_RECEIPT.json"
if receipt_path.is_file():
    receipt=json.loads(receipt_path.read_text(encoding="utf-8"))
    print("G1_V2_PREFLIGHT_RECEIPT="+json.dumps(receipt,ensure_ascii=False,sort_keys=True))
else:
    print("G1_V2_PREFLIGHT_RECEIPT=MISSING")
if preflight.returncode!=0:
    print("G1_V2_NEXT=STOP_AND_REVIEW_NO_MODEL_DOWNLOAD")
else:
    print("G1_V2_NEXT=PASS_PREFLIGHT_ONLY_OPTIONAL_SECOND_CELL_SMOKE")


In [ ]:
# G1 v2 — ONE technical smoke, explicit action in separate cell.
# Downloads Qwen weights (~1.5GB) only if the updated preflight passed.
import hashlib, json, pathlib, subprocess, sys
sroot=pathlib.Path("/content/ttm_six_g1")
script=sroot/"g1_qwen06_zero_science_smoke_colab_env_20261109_v2.py"
expected_blob="c98811e167696f86e94e3da9d01f3b9f020c3cc6"
if not script.is_file():
    raise RuntimeError("STOP_SCRIPT_MISSING_RUN_FIRST_CELL")
src=script.read_bytes()
actual_blob=hashlib.sha1(b"blob "+str(len(src)).encode()+b"\0"+src).hexdigest()
if actual_blob!=expected_blob:
    raise RuntimeError("STOP_SCRIPT_BLOB_CHANGED")
preflight_receipt=sroot/"G1_PREFLIGHT_RECEIPT.json"
if not preflight_receipt.is_file():
    raise RuntimeError("STOP_PREFLIGHT_RECEIPT_MISSING")
pre=json.loads(preflight_receipt.read_text(encoding="utf-8"))
if pre.get("status")!="PASS_PREFLIGHT_ONLY" or pre.get("environment_amendment")!="COLAB_RUNTIME_20261109_V2":
    raise RuntimeError("STOP_PREFLIGHT_NOT_PASS_FOR_V2")
env=pre.get("environment",{})
if env.get("python")!="3.13.16" or env.get("torch")!="2.14.1+cu130" or env.get("gpu_name")!="Tesla T4":
    raise RuntimeError("STOP_ENVIRONMENT_CHANGED_SINCE_PREFLIGHT")
once=sroot/"G1_SMOKE_ATTEMPTED_V2.marker"
with once.open("x",encoding="utf-8") as f:
    f.write("ONE_SHOT_CONSUMED_BEFORE_MODEL_DOWNLOAD\n")
try:
    run=subprocess.run([sys.executable,str(script),"--phase","smoke"],check=False)
    print("G1_V2_SMOKE_EXIT="+str(run.returncode))
    path=sroot/"G1_SMOKE_RECEIPT.json"
    print("G1_V2_SMOKE_RECEIPT="+
          (path.read_text(encoding="utf-8") if path.is_file() else "NOT_WRITTEN"))
    print("G1_V2_FINAL="+("PASS_TECHNICAL_ONLY" if run.returncode==0 else "STOP_REVIEW_RECEIPT"))
finally:
    print("G1_V2_ONE_SHOT_MARKER=CONSUMED_DO_NOT_RERUN")
